In [1]:
import wandb
import matplotlib.pyplot as plt
import numpy as np
import os

def setup_aaai_style():
    plt.rcParams.update({
        'font.size': 9,
        'axes.labelsize': 9,
        'legend.fontsize': 8,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'font.family': 'serif',
        'axes.grid': True,
        'axes.axisbelow': True,
        'grid.alpha': 0.3,
        'grid.linewidth': 0.8,
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.spines.left': True,
        'axes.spines.bottom': True,
        'axes.linewidth': 1.0,
        'xtick.direction': 'out',
        'ytick.direction': 'out',
        'lines.markersize': 4,
        'lines.linewidth': 1.2,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'axes.titlepad': 4,
    })

WANDB_TO_PAPER_NAME = {
    "UnsyncedRecurrentDifflogic": "RDDLGN",
    "UnsyncedGRU": "GRU",
    "UnsyncedRNN": "RNN",
    "Transformer": "Transformer"
}

def get_model_display_name(run_config):
    try:
        wandb_name = run_config['model']['value']['name']
    except Exception:
        try:
            wandb_name = run_config['model']['name']
        except Exception:
            wandb_name = run_config.get('model_name', 'UnknownModel')
    return WANDB_TO_PAPER_NAME.get(wandb_name, wandb_name)

def get_run_metrics(run_id):
    api = wandb.Api()
    run = api.run(run_id)
    summary = run.summary._json_dict if hasattr(run.summary, "_json_dict") else dict(run.summary)
    config = run.config
    display_name = get_model_display_name(config)
    results = {}

    for mode, label in zip(
        ["train", "val", "test"],
        ["Train", "Validation", "Test"]
    ):
        prefix = f"{mode}/metric/"
        bleu = summary.get(prefix + "bleu")
        perp = summary.get(prefix + "perplexity")
        acc = summary.get(prefix + "accuracy")
        if bleu is not None and perp is not None and acc is not None:
            results[label] = {
                "bleu": bleu,
                "perplexity": perp,
                "accuracy": acc,
            }

    prefix = "collapse_eval_col_all/metric/"
    bleu = summary.get(prefix + "bleu")
    perp = summary.get(prefix + "perplexity")
    acc = summary.get(prefix + "accuracy")
    if bleu is not None and perp is not None and acc is not None:
        results["Test (Collapsed)"] = {
            "bleu": bleu,
            "perplexity": perp,
            "accuracy": acc,
        }

    return display_name, results

def gather_all_metrics(run_ids):
    all_metrics = {"Train": {}, "Validation": {}, "Test": {}}
    for run_id in run_ids:
        run_ref = run_id if "/" in run_id else f"sbuehrer-eth-z-rich/RDDLGN/{run_id}"
        model_name, metrics = get_run_metrics(run_ref)
        for split in ["Train", "Validation", "Test"]:
            if split in metrics:
                all_metrics[split][model_name] = metrics[split]
        if "Test (Collapsed)" in metrics:
            all_metrics["Test"][f"{model_name} (Collapsed)"] = metrics["Test (Collapsed)"]

    return all_metrics

def shorten_label(label):
    if label.startswith("Transformer"):
        return "Tra."
    if label.startswith("RDDLGN"):
        return "RDDLGN*" if "Collapsed" in label else "RDDLGN"
    if label.startswith("GRU"):
        return "GRU"
    if label.startswith("RNN"):
        return "RNN"
    return label[:7]

def multibar_plot(metrics_dict, metric_names, split_label, output_dir):
    setup_aaai_style()
    models_sorted = sorted(metrics_dict.keys(), key=lambda m: metrics_dict[m]["perplexity"], reverse=True)
    models_sorted = models_sorted[::-1]

    n_models = len(models_sorted)
    n_metrics = len(metric_names)
    bar_width = 0.18
    x = np.arange(n_models)
    colors = ["#1f77b4", "#9467bd", "#ff7f0e"]

    fig, ax = plt.subplots(figsize=(3.37, 2.5), dpi=300)
    max_val = 0
    for i, metric in enumerate(metric_names):
        values = [metrics_dict[m][metric] for m in models_sorted]
        if metric == "accuracy" or metric == "bleu":
            values = [100 * v for v in values]
        max_val = max(max_val, max(values))
        pos = x + (i - 1) * bar_width
        bars = ax.bar(
            pos, values, bar_width, label=metric.capitalize(),
            color=colors[i % len(colors)], edgecolor="black", alpha=0.85, zorder=3
        )
        for xx, val in zip(pos, values):
            ax.text(
                xx, val * 1.07 if ax.get_yscale() == "log" else val + max_val * 0.07, f"{val:.1f}",
                ha='center', va='bottom', fontsize=8, rotation=90
            )

    xtick_labels = [shorten_label(m) for m in models_sorted]
    ax.set_xticks(x)
    ax.set_xticklabels(xtick_labels, rotation=12, ha='right', fontsize=9)

    # Set y axis to logarithmic and fix upper limit to 150
    ax.set_yscale("log")
    all_percent = []
    if "accuracy" in metric_names:
        all_percent += [100 * metrics_dict[m]["accuracy"] for m in models_sorted]
    if "bleu" in metric_names:
        all_percent += [100 * metrics_dict[m]["bleu"] for m in models_sorted]
    all_perplex = [metrics_dict[m]["perplexity"] for m in models_sorted]
    # Minimum y limit must be > 0 for log scale
    miny = min([v for v in all_percent + all_perplex if v > 0]) * 0.7 if (all_percent + all_perplex) else 0.1
    ax.set_ylim(miny, 700)

    ax.set_ylabel("Score", fontweight='bold', fontsize=9, labelpad=2)
    ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(0.0, 1.0), ncol=1, fontsize=8)
    plt.tight_layout(pad=0.22, rect=[0,0,1,1])

    os.makedirs(output_dir, exist_ok=True)
    fname = f"{output_dir}/barplot_{split_label.replace(' ', '_').lower()}_halfpage.pdf"
    fig.savefig(fname, format="pdf", dpi=300, bbox_inches="tight", pad_inches=0.03)
    fname = f"{output_dir}/barplot_{split_label.replace(' ', '_').lower()}_halfpage.png"

    fig.savefig(fname, format="png", bbox_inches="tight", pad_inches=0.03)

    print(f"Figure saved: {fname}")
    plt.close(fig)

def main():
    run_ids = [
        "u6fxkjv6", # UnsyncedRecurrentDifflogic (RDDLGN)
        "bfn0q6b0", # UnsyncedGRU (GRU)
        "283pnkzo", # UnsyncedRNN (RNN)
        "68s3tls8"  # Transformer (Transformer)
    ]
    all_metrics = gather_all_metrics(run_ids)
    metric_names = ["bleu", "perplexity", "accuracy"]
    outdir = "aaai_barplots"
    for split in ["Train", "Validation", "Test"]:
        if all_metrics[split]:
            multibar_plot(all_metrics[split], metric_names, split, outdir)

if __name__ == "__main__":
    main()

Figure saved: aaai_barplots/barplot_train_halfpage.png
Figure saved: aaai_barplots/barplot_validation_halfpage.png
Figure saved: aaai_barplots/barplot_test_halfpage.png
